# Step-by-Step Analysis Walkthrough

This notebook shows how the analysis summarized in Appendix B was carried out, from the raw email files to the results reported in Table 2. Each step describes what was done, and the cells that follow run the study's own code or read its saved output files and show the result directly below. Nothing is typed in by hand. The output of each step is the input of the next. The steps follow this sequence.

**Raw emails → (1) cleaned emails → (2) enhanced phishing emails → (3) paired dataset → (4) training and test sets → (5) d′ before and after enhancement → (6) 500 paired changes → (7) t-tests for RQ1 and RQ2 → (8) RQ3 and Table 2**

To run it, open this notebook from the main folder of the reproducibility package and choose **Run All**. The whole notebook takes under a minute. The first cell loads the study's code and a few display helpers. It is setup only and is not part of the analysis.

In [1]:
import importlib.util, difflib, html, inspect
from pathlib import Path
import pandas as pd
from IPython.display import display, HTML

PACKAGE = Path.cwd()
if not (PACKAGE / "run_all.py").exists():
    raise SystemExit("Open this notebook from the main folder of the reproducibility package.")

# Step 1 reads the stage counts from a rebuilt dataset: the folder created by
#   python run_all.py --stage data --workspace <folder>
# Leave None to use the counts saved from the study's rebuild (data/final/Dataset_Preparation_Counts.csv).
WORKSPACE = None


def load(name, path):
    spec = importlib.util.spec_from_file_location(name, PACKAGE / path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


rq1 = load("rq1", "code/rq_scripts/RQ1_classical_ml/scripts/run_rq1_classical_ml.py")
splits = load("splits", "code/rq_scripts/splits.py")
enhance = load("enhance", "code/scripts/nazario/06_enhance_nazario_with_openai_qc.py")
printer = load("print_results", "print_results.py")
RESULTS = PACKAGE / "results"
FINAL = PACKAGE / "data" / "final" / "Final_Paired_Email_Dataset.csv"


def show(table):
    """Display a table without the row-number column."""
    display(HTML(table.to_html(index=False)))


def count(file, key, column="count"):
    """Read one count from a summary file written while the dataset was rebuilt (or from the saved counts)."""
    if WORKSPACE is None:
        saved = pd.read_csv(PACKAGE / "data" / "final" / "Dataset_Preparation_Counts.csv")
        match = (saved["file"] == file) & (saved["key"] == key) & (saved["column"] == column)
        return int(saved.loc[match, "value"].iloc[0])
    t = pd.read_csv(Path(WORKSPACE) / "processed" / file)
    return int(float(t.loc[t.iloc[:, 0] == key, column].iloc[0]))


def show_changes(original, enhanced):
    """Show an original and enhanced email, and the words that changed."""
    a, b = original.split(), enhanced.split()
    parts = []
    for op, a1, a2, b1, b2 in difflib.SequenceMatcher(None, a, b).get_opcodes():
        if op == "equal":
            parts.append(html.escape(" ".join(a[a1:a2])))
        if op in ("delete", "replace"):
            parts.append('<del style="color:#b00020">' + html.escape(" ".join(a[a1:a2])) + "</del>")
        if op in ("insert", "replace"):
            parts.append('<b style="color:#00701a">' + html.escape(" ".join(b[b1:b2])) + "</b>")
    box = "border:1px solid #bbb;padding:8px 10px;margin:4px 0 12px;max-width:900px;line-height:1.5"
    display(HTML(f"<b>Original</b><div style='{box}'>{html.escape(original)}</div>"
                 f"<b>Enhanced</b><div style='{box}'>{html.escape(enhanced)}</div>"
                 f"<b>Changes</b> (removed in red, added in green)<div style='{box}'>{' '.join(parts)}</div>"))


def emails(units, variant):
    """The emails of one variant (legitimate, original_phishing, enhanced_phishing) in a set of split units."""
    label = 0 if variant == "legitimate" else 1
    ids = set(units.loc[units["label_binary"] == label, "pair_id"])
    return master[master["pair_id"].isin(ids) & (master["variant"] == variant)]


def corrected_test(name, differences, saved_file):
    """Run the study's corrected resampled t-test and compare it with the saved result."""
    r = rq1.corrected_resampled_t_test(differences, 0.30)
    saved = pd.read_csv(saved_file).query("metric == 'd_prime'").iloc[0]
    return {"Test": name, "Iterations": r["n_observations"], "Correction": round(r["correction"], 6),
            "Mean change": round(r["mean_difference"], 3),
            "95% CI": f"[{r['ci95_low']:.3f}, {r['ci95_high']:.3f}]",
            "t": round(r["t_statistic"], 2), "df": r["degrees_of_freedom"],
            "p": round(r["p_value_two_sided"], 3),
            "Same as saved": "yes" if abs(r["p_value_two_sided"] - saved["p_value_two_sided"]) < 1e-9 else "no"}


print("Package folder:", PACKAGE.name)

Package folder: DSRT930_Reproducibility_Package


## Step 1. Data preparation

The raw Enron Email Dataset (legitimate emails) and Nazario Phishing Corpus (phishing emails) were parsed and filtered in stages, as described in Chapter 3. Figure E1 shows the number of emails retained at each stage, read from the summary files written by the code at each stage. These are the counts reported in Table C1. The 3,508 phishing emails remaining after the final data quality check were passed to Step 2.

The summary files are created when the dataset is rebuilt from the raw files with `python run_all.py --stage data --workspace <folder>` (about 15 minutes). To read the counts from your own rebuild, set `WORKSPACE` in the first cell to that folder. Otherwise, the counts saved from the study's rebuild (`data/final/Dataset_Preparation_Counts.csv`) are used.

**Figure E1. Dataset preparation counts (Table C1)**

In [2]:
qc = "dissertation_aligned_source_qc/Dissertation_Aligned_SourceQC_Counts.csv"
final = "dissertation_aligned_source_qc/Final_Paired_Email_Dataset_Quality_Audit.csv"
stages = [
    ("Initially parsed messages",
     count("enron/Enron_FULL_Eligibility_Summary.csv", "parsed_records"),
     count("nazario/Nazario_Initial_Audit_Summary.csv", "parsed_messages")),
    ("Retained after date, standalone, content, and 25-500-word criteria",
     count("enron/Enron_FULL_Dedup_Summary.csv", "final_candidate_after_length_filter"),
     count("nazario/Nazario_Initial_Audit_Summary.csv", "prelim_standalone_pre2020_candidate")),
    ("Retained after exact-duplicate removal",
     count("enron/Enron_FULL_Dedup_Summary.csv", "final_candidate_after_dedup"),
     count("nazario/Nazario_Initial_Audit_Summary.csv", "prelim_candidate_after_dedup")),
    ("Retained after text cleaning",
     count("enron/Enron_Modeling_Text_Artifact_Removal_Audit.csv", "final_modeling_rows", "before_rows"),
     count("nazario/Nazario_Cleaning_Transformation_Summary.csv", "final_cleaned_candidates")),
    ("Retained after near-duplicate removal and invalid-record screening",
     count(qc, "enron_strict_final", "final_095_modeling_records"),
     count(qc, "nazario", "final_095_modeling_records")),
    ("Retained after final data quality check",
     count(final, "final_legitimate_rows", "value"), count(final, "phishing_queue_rows", "value")),
    ("Phishing emails enhanced and passing the enhancement quality checks",
     "-", f'{count(final, "accepted_enhanced_phishing_rows", "value"):,} pairs'),
    ("Excluded at final build (character-level obfuscation and duplicate text)",
     0, f'{count(final, "phishing_pairs_excluded", "value"):,} pairs'),
    ("Final analysis dataset",
     count(final, "final_legitimate_rows", "value"), f'{count(final, "complete_phishing_pairs", "value"):,} pairs'),
]
table = pd.DataFrame(stages, columns=["Stage", "Legitimate emails", "Phishing emails"])
show(table.map(lambda v: f"{v:,}" if isinstance(v, int) else v))

Stage,Legitimate emails,Phishing emails
Initially parsed messages,"102,512","10,201"
"Retained after date, standalone, content, and 25-500-word criteria","24,972","8,939"
Retained after exact-duplicate removal,"22,262","6,742"
Retained after text cleaning,"19,994","5,135"
Retained after near-duplicate removal and invalid-record screening,"19,614","3,566"
Retained after final data quality check,"19,393","3,508"
Phishing emails enhanced and passing the enhancement quality checks,-,"3,497 pairs"
Excluded at final build (character-level obfuscation and duplicate text),0,20 pairs
Final analysis dataset,"19,393","3,477 pairs"


## Step 2. LLM enhancement

Each of the 3,508 phishing emails was sent to GPT-5.4-mini with the system message and user prompt shown in Figure E2, read directly from the enhancement script. Of these, 3,497 enhanced versions passed the enhancement quality checks. Figure E3 shows one original email, its enhanced version, and the words that were changed.

**Figure E2. Enhancement model, system message, and user prompt**

In [3]:
print("Model:         ", enhance.DEFAULT_MODEL)
print("System message:", enhance.SYSTEM_MESSAGES["B"])
print("User prompt:   ", enhance.PROMPT)

Model:          gpt-5.4-mini-2026-03-17
System message: You edit email text only. Correct grammar and improve fluency only; do not add, remove, or change any other content. Return only the revised email body.
User prompt:    Please correct any grammatical errors and improve the fluency of the following email without changing its intent.


**Figure E3. Example of an original and enhanced phishing email**

In [4]:
data = pd.read_csv(FINAL, keep_default_na=False)
pair = data[data["pair_id"] == "nazario_001901"].set_index("variant")
show_changes(pair.loc["original_phishing", "body_model"], pair.loc["enhanced_phishing", "body_model"])

## Step 3. Paired dataset

Each original phishing email was linked to its enhanced version by a shared pair ID. After 20 pairs were excluded at the final build, the analysis dataset contained 19,393 legitimate emails and 3,477 phishing pairs (Figure E4). Each email was also assigned to a template family, a group of near-identical emails such as copies of the same phishing campaign. Both versions of a phishing email belong to the same family.

**Figure E4. Paired dataset**

In [5]:
print(f"{len(data):,} rows in Final_Paired_Email_Dataset.csv")
show(data.groupby(["label", "variant"]).size().reset_index(name="rows"))
columns = ["pair_id", "label", "variant", "template_family_id", "body_model"]
example = pd.concat([data[data["variant"] == "legitimate"].head(2), data[data["pair_id"] == "nazario_001901"]])
show(example[columns].assign(body_model=example["body_model"].str.slice(0, 60) + "..."))

26,347 rows in Final_Paired_Email_Dataset.csv


label,variant,rows
legitimate,legitimate,19393
phishing,enhanced_phishing,3477
phishing,original_phishing,3477


pair_id,label,variant,template_family_id,body_model
enron_000001,legitimate,legitimate,legit_tf_000001,The attached list are outstanding receivables associated wit...
enron_000002,legitimate,legitimate,legit_tf_000002,Mike/Steve - we have verified all of the volume information ...
nazario_001901,phishing,original_phishing,phish_tf_001254,"Dear Visa client, Security and confidentiality areat the hea..."
nazario_001901,phishing,enhanced_phishing,phish_tf_001254,"Dear Visa client, Security and confidentiality are at the he..."


## Step 4. Training and test sets

In each of 500 iterations, the dataset was split into a training set (70%) and a test set (30%). The first iteration used a random seed of 930, the second 931, and so on. Whole template families were kept on one side of the split, so near-copies of a test email never appeared in the training set. Figure E5 shows the split for the first iteration, made with the study's own split function.

**Figure E5. Grouped split for the first iteration (seed 930)**

In [6]:
master, legitimate_ids, phishing_ids = rq1.prepare_master(FINAL)
units = splits.unit_table(master, legitimate_ids, phishing_ids)
SEED = 930
train_units, test_units = splits.make_split(units, "grouped", 0.30, SEED)

sizes = pd.DataFrame({"Training": train_units["label_binary"].value_counts(),
                      "Test": test_units["label_binary"].value_counts()}).sort_index()
sizes.insert(0, "Emails", ["Legitimate emails", "Phishing pairs"])
sizes["Test share"] = (sizes["Test"] / (sizes["Training"] + sizes["Test"])).map("{:.1%}".format)
show(sizes)
shared = set(train_units["template_family_id"]) & set(test_units["template_family_id"])
print("Template families in both training and test sets:", len(shared))

Emails,Training,Test,Test share
Legitimate emails,13575,5818,30.0%
Phishing pairs,2434,1043,30.0%


Template families in both training and test sets: 0


## Step 5. d′ before and after enhancement

Logistic regression, linear SVM, and random forest models were trained on the legitimate and original phishing emails in the training set. Each model was then tested twice on the same test emails, first with the original phishing emails (baseline) and then with their enhanced versions (enhanced). The legitimate test emails were identical in both. d′ measures how well a model separates phishing from legitimate emails, and Figure E6 shows the code that computes it. The difference between the baseline and enhanced d′ is the change attributable to the enhancement. Figure E7 shows the three models retrained for the first iteration, producing the same d′ values as those saved in the study's results.

**Figure E6. Code that computes d′**

In [7]:
print(inspect.getsource(rq1.d_prime_from_predictions))

def d_prime_from_predictions(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    hits = int(((y_true == 1) & (y_pred == 1)).sum())
    misses = int(((y_true == 1) & (y_pred == 0)).sum())
    false_alarms = int(((y_true == 0) & (y_pred == 1)).sum())
    correct_rejections = int(((y_true == 0) & (y_pred == 0)).sum())

    signal_n = hits + misses
    noise_n = false_alarms + correct_rejections
    hit_rate = (hits + 0.5) / (signal_n + 1.0)
    false_alarm_rate = (false_alarms + 0.5) / (noise_n + 1.0)
    return float(stats.norm.ppf(hit_rate) - stats.norm.ppf(false_alarm_rate))



**Figure E7. Classical models retrained for the first iteration**

In [8]:
train = pd.concat([emails(train_units, "legitimate"), emails(train_units, "original_phishing")])
baseline_test = pd.concat([emails(test_units, "legitimate"), emails(test_units, "original_phishing")])
enhanced_test = pd.concat([emails(test_units, "legitimate"), emails(test_units, "enhanced_phishing")])
saved = pd.read_csv(RESULTS / "RQ1_classical_ml/primary_grouped_split/RQ1_iteration_model_results.csv")
saved = saved[saved["iteration"] == 1].set_index("model")

rows = []
for name, model in rq1.build_models(SEED, 50000, 100).items():
    model.fit(train["body_model"], train["label_binary"])
    baseline = rq1.d_prime_from_predictions(baseline_test["label_binary"].to_numpy(), model.predict(baseline_test["body_model"]))
    enhanced = rq1.d_prime_from_predictions(enhanced_test["label_binary"].to_numpy(), model.predict(enhanced_test["body_model"]))
    same = abs(baseline - saved.loc[name, "baseline_d_prime"]) < 1e-9 and abs(enhanced - saved.loc[name, "enhanced_d_prime"]) < 1e-9
    rows.append({"Model": name, "Baseline d'": round(baseline, 4), "Enhanced d'": round(enhanced, 4),
                 "Change": round(baseline - enhanced, 4), "Same as saved": "yes" if same else "no"})
print(f"Training emails: {len(train):,}   Test emails: {len(baseline_test):,} (baseline) and {len(enhanced_test):,} (enhanced)")
show(pd.DataFrame(rows))

Training emails: 16,009   Test emails: 6,861 (baseline) and 6,861 (enhanced)


Model,Baseline d',Enhanced d',Change,Same as saved
logistic_regression,4.5245,4.5245,0.0000,yes
linear_svm,4.8972,4.8600,0.0372,yes
random_forest,4.7678,4.7398,0.0280,yes


## Step 6. 500 paired changes

Steps 4 and 5 were repeated for all 500 iterations. RQ2 followed the same process with LSTM, GRU, and CNN models, trained on a GPU in Google Colab (RQ2_on_Colab.ipynb). In each iteration, the changes in d′ of the three models in each family were averaged, producing one change per family per iteration. Figure E8 shows the first rows of these results.

**Figure E8. Per-iteration results**

In [9]:
rq1_models = pd.read_csv(RESULTS / "RQ1_classical_ml/primary_grouped_split/RQ1_iteration_model_results.csv")
rq1_family = pd.read_csv(RESULTS / "RQ1_classical_ml/primary_grouped_split/RQ1_family_iteration_results.csv")
rq2_family = pd.read_csv(RESULTS / "RQ2_deep_learning/primary_grouped_split/RQ2_family_iteration_results.csv")
rq2_audit = pd.read_csv(RESULTS / "RQ2_deep_learning/primary_grouped_split/RQ2_analysis_audit.csv").set_index("item")["value"]

print("RQ1, one row per model and iteration (first 6 rows)")
show(rq1_models[["iteration", "model", "baseline_d_prime", "enhanced_d_prime",
                 "d_prime_difference_baseline_minus_enhanced"]].head(6).round(4))
changes = pd.DataFrame({"iteration": rq1_family["iteration"],
                        "Classical ML change": rq1_family["family_d_prime_difference_baseline_minus_enhanced"],
                        "Deep learning change": rq2_family["family_d_prime_difference_baseline_minus_enhanced"]})
print(f"Family change in d' per iteration (first 5 of {len(changes)} iterations)")
show(changes.head(5).round(4))
print(f"RQ2 trained on {rq2_audit['device']} (GPU), PyTorch {rq2_audit['torch_version']}, "
      f"same dataset as RQ1 (SHA-256 {rq2_audit['master_dataset_sha256'][:12]}...)")

RQ1, one row per model and iteration (first 6 rows)


iteration,model,baseline_d_prime,enhanced_d_prime,d_prime_difference_baseline_minus_enhanced
1,logistic_regression,4.5245,4.5245,0.0000
1,linear_svm,4.8972,4.8600,0.0372
1,random_forest,4.7678,4.7398,0.0280
2,logistic_regression,4.5724,4.5724,0.0000
2,linear_svm,4.9679,4.9520,0.0159
2,random_forest,4.5802,4.5874,-0.0071


Family change in d' per iteration (first 5 of 500 iterations)


iteration,Classical ML change,Deep learning change
1,0.0217,0.0125
2,0.0029,-0.0018
3,-0.0013,0.0181
4,0.0000,0.0301
5,0.0196,0.0075


RQ2 trained on cuda (GPU), PyTorch 2.11.0+cu130, same dataset as RQ1 (SHA-256 25dcf2247880...)


## Step 7. Corrected resampled t-tests for RQ1 and RQ2

Because the 500 iterations reuse the same emails, their results are not independent. The corrected resampled t-test accounts for this by enlarging the variance by a correction factor of 1/500 + 30/70 = 0.430571 (Nadeau & Bengio, 2003). Figure E9 shows the code of the test and its results for the 500 classical ML and 500 deep learning changes. These are the RQ1 and RQ2 results reported in Chapter 4.

**Figure E9. Corrected resampled t-test code and RQ1 and RQ2 results**

In [10]:
key_lines = ("def ", "values =", "n =", "mean_diff =", "sample_var =", "train_size =", "correction =",
             "standard_error =", "t_statistic =", "p_value =", "half_width =")
source = inspect.getsource(rq1.corrected_resampled_t_test).splitlines()
print("Key lines of the study's function:\n")
print("\n".join(line for line in source if line.strip().startswith(key_lines)))
print()
show(pd.DataFrame([
    corrected_test("RQ1: Classical ML", changes["Classical ML change"],
                   RESULTS / "RQ1_classical_ml/primary_grouped_split/RQ1_corrected_resampled_t_tests.csv"),
    corrected_test("RQ2: Deep learning", changes["Deep learning change"],
                   RESULTS / "RQ2_deep_learning/primary_grouped_split/RQ2_corrected_resampled_t_tests.csv"),
]))

Key lines of the study's function:

def corrected_resampled_t_test(differences: pd.Series, test_size: float) -> dict[str, float]:
    values = differences.dropna().astype(float)
    n = len(values)
    mean_diff = float(values.mean())
    sample_var = float(values.var(ddof=1))
    train_size = 1.0 - test_size
    correction = (1.0 / n) + (test_size / train_size)
    standard_error = math.sqrt(correction * sample_var)
    t_statistic = mean_diff / standard_error
    p_value = float(2 * stats.t.sf(abs(t_statistic), df=n - 1))
    half_width = float(stats.t.ppf(0.975, df=n - 1)) * standard_error



Test,Iterations,Correction,Mean change,95% CI,t,df,p,Same as saved
RQ1: Classical ML,500,0.430571,0.004,"[-0.015, 0.024]",0.44,499,0.662,yes
RQ2: Deep learning,500,0.430571,0.012,"[-0.015, 0.038]",0.87,499,0.383,yes


## Step 8. RQ3 and Table 2

For RQ3, the deep learning change was subtracted from the classical ML change in each iteration, and the same test was applied to these 500 differences. The four sensitivity analyses (random split, masked identifiers, with bulk emails, and without repeated-passage pairs) repeated Steps 4 to 7 and the RQ3 comparison with one change each, using 100 iterations. Figure E10 shows the RQ3 result and Table 2, built from the saved results of all five analyses.

**Figure E10. RQ3 result and Table 2**

In [11]:
show(pd.DataFrame([corrected_test("RQ3: Classical ML - Deep learning",
                                  changes["Classical ML change"] - changes["Deep learning change"],
                                  RESULTS / "RQ3_model_family_comparison/primary_grouped_split/RQ3_corrected_resampled_t_tests.csv")]))

results = printer.Source()
table2 = [{"Analysis": label, "Iterations": results.ttest("RQ1", folder, "d_prime")["n_observations"],
           **{rq: printer.change_text(results.ttest(rq, folder, "d_prime")) for rq in ("RQ1", "RQ2", "RQ3")}}
          for folder, label in printer.ANALYSES]
print("\nTable 2. Change in d' (baseline - enhanced), 95% CI, and p from corrected resampled t-tests")
show(pd.DataFrame(table2))

Test,Iterations,Correction,Mean change,95% CI,t,df,p,Same as saved
RQ3: Classical ML - Deep learning,500,0.430571,-0.007,"[-0.038, 0.023]",-0.48,499,0.633,yes



Table 2. Change in d' (baseline - enhanced), 95% CI, and p from corrected resampled t-tests


Analysis,Iterations,RQ1,RQ2,RQ3
Primary (grouped split),500,"0.004 [-0.015, 0.024], p = .662","0.012 [-0.015, 0.038], p = .383","-0.007 [-0.038, 0.023], p = .633"
Random split,100,"-0.001 [-0.021, 0.019], p = .915","0.010 [-0.013, 0.034], p = .388","-0.011 [-0.040, 0.017], p = .435"
Masked identifiers,100,"0.008 [-0.012, 0.029], p = .416","0.019 [-0.008, 0.046], p = .166","-0.011 [-0.044, 0.023], p = .526"
With bulk emails,100,"0.003 [-0.019, 0.026], p = .766","0.011 [-0.016, 0.037], p = .436","-0.007 [-0.040, 0.026], p = .666"
Without repeated-passage pairs,100,"0.005 [-0.017, 0.027], p = .643","0.010 [-0.014, 0.034], p = .422","-0.005 [-0.035, 0.025], p = .750"
